# 02 · Long context: XLM-R Longformer on Jigsaw Toxic Comment
Buổi 13. `AshtonIsNotHere/xlm-roberta-long-base-4096` reads up to **4,096 tokens** in one pass, so long reviews are never cut (BERT-style models stop at 512).

Trained at 1,024 tokens to fit a T4, evaluated at 4,096. Reports macro ROC-AUC and the share of long (> 512 token) comments read whole.
Multilingual XLM-R weights let the toxicity check transfer to Vietnamese reviews. About 40 minutes on a T4.

In [ ]:
#@title Setup — run once per session (Runtime > Change runtime type > T4 GPU)
REPO_URL = "https://github.com/YoungsterDumb/reviewlens"  # @param {type:"string"}
HF_NAMESPACE = "DatDuc"  # @param {type:"string"}  your Hugging Face username
import os, sys, subprocess
if not os.path.exists("/content/reviewlens"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, "/content/reviewlens"], check=True)
else:
    subprocess.run(["git", "-C", "/content/reviewlens", "pull", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/reviewlens/training/requirements.txt"], check=True)
sys.path.insert(0, "/content/reviewlens/training")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none — switch the runtime to T4 GPU")

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # paste a token with WRITE access (huggingface.co/settings/tokens)
METRICS_REPO = f"{HF_NAMESPACE}/reviewlens-metrics"

In [ ]:
from pathlib import Path
from reviewlens_train.longctx_train import load_jigsaw, train_longformer, evaluate_longformer
frames = load_jigsaw(n_train=16_000)
{k: len(v) for k, v in frames.items()}

In [ ]:
trainer, tok = train_longformer(frames, Path("/content/out/longformer"))

In [ ]:
long_metrics = evaluate_longformer(trainer.model, tok, frames["test"])
long_metrics

In [ ]:
from reviewlens_train.hub import push_model, update_metrics
push_model(trainer.model, tok, f"{HF_NAMESPACE}/reviewlens-longctx", "Longformer toxicity (Jigsaw)")
update_metrics(METRICS_REPO, long_metrics)